# Lesson 6.3 — Failure Analysis: What / Where / Who Owns the Fix

Each fired event resolves to the three-question triad with a named owner. Where a fault surfaced can differ from who owns the fix.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, to_configuration, run_pipeline, localize, failure_event,
    FAILURE_TAXONOMY, REACH_MAX)
W = GreenhouseWorld.demo_row(n=6, seed=1)
ALL_IDS = [f.fid for f in W.fruit]
TGT = understand(model_perception(W, rng=np.random.default_rng(7)), W)[1]
def big(t, j): return 60.0 if (j == 0 and t > 0.3) else 0.0
checks = []
def analyse(r):
    e = r['events'][0]
    print('event:', e['code'], '| reached:', r['reached'])
    print(localize(e)); print()
    return e


### NO_TARGET -> owned by Perceive/Understand

In [2]:
e1 = analyse(run_pipeline(W, W.q.copy(), perception=dict(occlude=ALL_IDS)))
checks.append(e1['where']=='Understand' and 'Understand' in e1['who'])


event: NO_TARGET | reached: Understand
What failed?  No committable target
Where did it fail?  Understand
Who owns the fix?  Perceive / Understand (re-perceive or widen search)



### TRACKING_FAILURE -> owned by Execute/Recover (NOT Perceive or Plan)

In [3]:
e2 = analyse(run_pipeline(W, W.q.copy(), disturbance=big))
checks.append('Execute' in e2['who'] and 'Perceive' not in e2['who'])


event: TRACKING_FAILURE | reached: Track
What failed?  Execution missed the success criteria
Where did it fail?  Execute → Track
Who owns the fix?  Execute / Recover



### UNREACHABLE surfaces at the IK seam but is OWNED by Understand

In [4]:
uev = failure_event('UNREACHABLE')
print('UNREACHABLE  surfaced:', uev['where'], '| owner:', uev['who'])
checks.append('IK seam' in uev['where'] and 'Understand' in uev['who'])
# and in the integrated pipeline, an out-of-reach world pre-empts it as NO_TARGET at Understand
w_far = GreenhouseWorld(fruit=[Fruit('FX', REACH_MAX+0.3, 0.0, ripe=True)], q=np.array([0.3,0.6]))
rf = run_pipeline(w_far, w_far.q.copy())
print('out-of-reach world -> reached=%s event=%s (filter pre-empts)' % (rf['reached'], [e['code'] for e in rf['events']]))
checks.append(rf['reached']=='Understand' and rf['events'][0]['code']=='NO_TARGET')
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


UNREACHABLE  surfaced: Understand → Plan (IK seam) | owner: Understand (re-select a reachable target)
out-of-reach world -> reached=Understand event=['NO_TARGET'] (filter pre-empts)
4/4 checks passed.
All checks passed.
